In [1]:
%pip install -U yfinance pandas numpy matplotlib seaborn plotly scipy openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
%pip install -U plotly nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [3]:
import os
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import norm
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

sns.set_theme(style="whitegrid")

np.random.seed(42)

print("TechStock Insights initialized successfully!")

TechStock Insights initialized successfully!


In [4]:
TICKERS = ["AAPL", "GOOGL", "MSFT", "AMZN"]

START_DATE = "2015-01-01"
END_DATE = None
INTERVAL = "1d"

INITIAL_INVESTMENT = 10_000
TRADING_DAYS = 252
RISK_FREE_RATE = 0.0
VAR_CONFIDENCE = 0.95
NUM_SIMULATIONS = 1000

OUTPUT_FOLDER = Path("techstock_outputs")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print("Stocks:", TICKERS)
print("Start date:", START_DATE)
print("End date: latest available data")
print("Output folder:", OUTPUT_FOLDER.resolve())

Stocks: ['AAPL', 'GOOGL', 'MSFT', 'AMZN']
Start date: 2015-01-01
End date: latest available data
Output folder: C:\Users\hp\techstock_outputs


In [5]:
print("Fetching historical stock data...")

stock_data = yf.download(
    tickers=TICKERS,
    start=START_DATE,
    end=END_DATE,
    interval=INTERVAL,
    auto_adjust=True,
    group_by="column",
    progress=False,
    threads=True
)

if stock_data is None or stock_data.empty:
    raise RuntimeError(
        "Download failed. Check your internet connection "
        "and try running this cell again."
    )

# Extract adjusted closing prices
if isinstance(stock_data.columns, pd.MultiIndex):
    close_prices = stock_data["Close"].copy()
else:
    close_prices = stock_data[["Close"]].copy()
    close_prices.columns = [TICKERS[0]]

close_prices.columns = [
    str(col).upper() for col in close_prices.columns
]

available_tickers = [
    ticker for ticker in TICKERS
    if ticker in close_prices.columns
]

close_prices = close_prices[available_tickers]

close_prices.index = pd.to_datetime(close_prices.index)
close_prices.index.name = "Date"
close_prices = close_prices.sort_index()
close_prices = close_prices[
    ~close_prices.index.duplicated(keep="last")
]

close_prices = close_prices.apply(pd.to_numeric, errors="coerce")
close_prices = close_prices.dropna(how="all")

if close_prices.empty:
    raise ValueError("No closing prices were found.")

# Forward-fill isolated gaps for analysis
prices = close_prices.ffill().dropna(how="all")

# Daily percentage returns
returns = prices.pct_change(fill_method=None).dropna(how="all")

# Save downloaded data
stock_data.to_csv(OUTPUT_FOLDER / "historical_stock_data.csv")
prices.to_csv(OUTPUT_FOLDER / "cleaned_stock_prices.csv")
returns.to_csv(OUTPUT_FOLDER / "daily_returns.csv")

print("\nDownload successful!")
print("Stocks:", list(prices.columns))
print("First date:", prices.index.min().date())
print("Latest date:", prices.index.max().date())
print("Total trading observations:", len(prices))

display(prices.tail())

Fetching historical stock data...

Download successful!
Stocks: ['AAPL', 'GOOGL', 'MSFT', 'AMZN']
First date: 2015-01-02
Latest date: 2026-10-14
Total trading observations: 2963


,AAPL,GOOGL,MSFT,AMZN
Date,,,,
2026-10-08,340.420013,348.290009,522.609985,254.059998
2026-10-09,336.640015,351.660004,535.070007,262.429993
2026-10-12,340.970001,352.720001,541.150024,259.529999
2026-10-13,341.269989,353.779999,543.450012,258.579987
2026-10-14,344.825012,353.029999,537.080017,255.339996


In [6]:
print("Dataset shape:", prices.shape)
print("\nColumn names:", prices.columns.tolist())

print("\nMissing values:")
display(prices.isnull().sum().to_frame("Missing Values"))

print("\nDuplicate dates:", prices.index.duplicated().sum())

print("\nDescriptive statistics:")
display(prices.describe().round(2))

print("\nLatest available prices:")
display(prices.iloc[-1].to_frame("Latest Price (USD)"))

Dataset shape: (2963, 4)

Column names: ['AAPL', 'GOOGL', 'MSFT', 'AMZN']

Missing values:


,Missing Values
AAPL,0
GOOGL,0
MSFT,0
AMZN,0



Duplicate dates: 0

Descriptive statistics:


,AAPL,GOOGL,MSFT,AMZN
count,2963.00,2963.00,2963.00,2963.00
mean,118.38,109.97,220.03,120.46
std,85.61,83.19,149.39,68.20
min,20.55,24.62,34.22,14.35
25%,38.04,51.17,77.49,58.46
50%,117.15,85.73,205.63,109.65
75%,177.79,140.58,341.29,170.93
max,344.83,402.12,543.45,284.02



Latest available prices:


,Latest Price (USD)
AAPL,344.825012
GOOGL,353.029999
MSFT,537.080017
AMZN,255.339996


In [7]:
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)

fig = go.Figure()

for ticker in prices.columns:
    stock_prices = prices[ticker].dropna()

    if not stock_prices.empty:
        fig.add_trace(go.Scatter(
            x=stock_prices.index,
            y=stock_prices,
            mode="lines",
            name=ticker
        ))

fig.update_layout(
    title="TechStock Insights — Historical Stock Prices",
    xaxis_title="Date",
    yaxis_title="Adjusted Price (USD)",
    template="plotly_dark",
    hovermode="x unified",
    height=600
)

fig.show(renderer="iframe")

## Compare normalized stock performance
#### This puts every stock on the same starting value of 100, making their performance easier to compare.

In [8]:
def show_chart(fig):
    fig.show(renderer="iframe")

normalized_prices = prices.div(prices.iloc[0]).mul(100)

fig = go.Figure()

for ticker in normalized_prices.columns:
    series = normalized_prices[ticker].dropna()
    fig.add_trace(go.Scatter(
        x=series.index,
        y=series,
        mode="lines",
        name=ticker
    ))

fig.update_layout(
    title="TechStock Insights — Normalized Stock Performance",
    xaxis_title="Date",
    yaxis_title="Growth of $100 (starting value = 100)",
    template="plotly_dark",
    hovermode="x unified",
    height=600
)

show_chart(fig)

## Calculate daily returns
#### Daily returns show the percentage change in each stock's price from one trading day to the next.

In [9]:
returns = prices.pct_change(fill_method=None)
returns = returns.replace([np.inf, -np.inf], np.nan)
returns = returns.dropna(how="all")

display(returns.head())
print("Daily returns shape:", returns.shape)

,AAPL,GOOGL,MSFT,AMZN
Date,,,,
2015-01-05,-0.028172,-0.019054,-0.009196,-0.020517
2015-01-06,0.000094,-0.024679,-0.014677,-0.022833
2015-01-07,0.014022,-0.002941,0.012706,0.010600
2015-01-08,0.038422,0.003484,0.029418,0.006836
2015-01-09,0.001073,-0.012211,-0.008406,-0.011749


Daily returns shape: (2962, 4)


## Visualize daily returns

In [10]:
fig = go.Figure()

for ticker in returns.columns:
    series = returns[ticker].dropna()
    fig.add_trace(go.Scatter(
        x=series.index,
        y=series * 100,
        mode="lines",
        name=ticker
    ))

fig.update_layout(
    title="Daily Stock Returns",
    xaxis_title="Date",
    yaxis_title="Daily Return (%)",
    template="plotly_dark",
    hovermode="x unified",
    height=600
)

show_chart(fig)

## Calculate cumulative returns

In [11]:
cumulative_returns = (1 + returns.fillna(0)).cumprod() - 1

fig = go.Figure()

for ticker in cumulative_returns.columns:
    fig.add_trace(go.Scatter(
        x=cumulative_returns.index,
        y=cumulative_returns[ticker] * 100,
        mode="lines",
        name=ticker
    ))

fig.update_layout(
    title="Cumulative Stock Returns",
    xaxis_title="Date",
    yaxis_title="Cumulative Return (%)",
    template="plotly_dark",
    hovermode="x unified",
    height=600
)

show_chart(fig)

## Moving averages
#### The 20-day and 50-day moving averages smooth out short-term price fluctuations.

In [12]:
ticker_to_analyze = "AAPL"

if ticker_to_analyze not in prices.columns:
    ticker_to_analyze = prices.columns[0]

stock = prices[ticker_to_analyze].dropna()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=stock.index,
    y=stock,
    mode="lines",
    name="Actual Price"
))

fig.add_trace(go.Scatter(
    x=stock.index,
    y=stock.rolling(20).mean(),
    mode="lines",
    name="20-Day Moving Average"
))

fig.add_trace(go.Scatter(
    x=stock.index,
    y=stock.rolling(50).mean(),
    mode="lines",
    name="50-Day Moving Average"
))

fig.update_layout(
    title=f"{ticker_to_analyze} — Price and Moving Averages",
    xaxis_title="Date",
    yaxis_title="Adjusted Price (USD)",
    template="plotly_dark",
    hovermode="x unified",
    height=600
)

show_chart(fig)

## Stock correlation heatmap
#### Correlation measures how similarly stocks' daily returns move. Values closer to 1 indicate stronger positive relationships.

In [13]:
correlation_matrix = returns.corr()

fig = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=correlation_matrix.columns,
    y=correlation_matrix.index,
    zmin=-1,
    zmax=1,
    colorscale="RdBu",
    text=np.round(correlation_matrix.values, 2),
    texttemplate="%{text}",
    colorbar_title="Correlation"
))

fig.update_layout(
    title="Correlation Between Tech Stocks",
    template="plotly_dark",
    height=500
)

show_chart(fig)

## Annualized volatility
#### Volatility estimates how much a stock's daily returns fluctuate. Higher volatility generally indicates greater price variability.

In [14]:
annualized_volatility = (
    returns.std() * np.sqrt(TRADING_DAYS) * 100
).sort_values(ascending=False)

volatility_table = annualized_volatility.rename(
    "Annualized Volatility (%)"
).to_frame()

display(volatility_table.round(2))

fig = go.Figure(go.Bar(
    x=annualized_volatility.index,
    y=annualized_volatility.values,
    text=[f"{v:.2f}%" for v in annualized_volatility.values],
    textposition="outside"
))

fig.update_layout(
    title="Annualized Stock Volatility",
    xaxis_title="Stock",
    yaxis_title="Annualized Volatility (%)",
    template="plotly_dark",
    height=500
)

show_chart(fig)

,Annualized Volatility (%)
AMZN,33.04
GOOGL,29.03
AAPL,28.67
MSFT,27.55


## Maximum drawdown
#### Drawdown measures the decline from a previous price peak. Maximum drawdown is the largest such decline during the selected period.

In [15]:
running_peak = prices.cummax()
drawdowns = prices / running_peak - 1
maximum_drawdown = drawdowns.min() * 100

drawdown_table = maximum_drawdown.rename(
    "Maximum Drawdown (%)"
).to_frame().sort_values("Maximum Drawdown (%)")

display(drawdown_table.round(2))

fig = go.Figure()

for ticker in drawdowns.columns:
    fig.add_trace(go.Scatter(
        x=drawdowns.index,
        y=drawdowns[ticker] * 100,
        mode="lines",
        name=ticker
    ))

fig.update_layout(
    title="Stock Drawdowns From Previous Peaks",
    xaxis_title="Date",
    yaxis_title="Drawdown (%)",
    template="plotly_dark",
    hovermode="x unified",
    height=600
)

show_chart(fig)

,Maximum Drawdown (%)
AMZN,-56.15
GOOGL,-44.32
AAPL,-38.52
MSFT,-37.15


## Historical Value at Risk (VaR)
#### Historical VaR estimates a loss threshold from past daily returns. At a 95% confidence level, it estimates the daily loss threshold that was exceeded on approximately 5% of historical days.

In [16]:
var_losses = (
    -returns.quantile(1 - VAR_CONFIDENCE) * 100
).rename("Historical VaR (%)")

display(var_losses.to_frame().round(3))

,Historical VaR (%)
AAPL,2.707
GOOGL,2.693
MSFT,2.657
AMZN,3.061


#### A larger positive VaR value means a larger estimated daily loss threshold. This is a historical estimate, not a guarantee or worst-case loss.

# Risk summary

In [17]:
risk_summary = pd.DataFrame({
    "Average Daily Return (%)": returns.mean() * 100,
    "Annualized Volatility (%)": returns.std() * np.sqrt(TRADING_DAYS) * 100,
    "Maximum Drawdown (%)": prices.div(prices.cummax()).sub(1).min() * 100,
    f"Historical VaR {VAR_CONFIDENCE:.0%} (%)": -returns.quantile(1 - VAR_CONFIDENCE) * 100
})

display(risk_summary.round(3))

,Average Daily Return (%),Annualized Volatility (%),Maximum Drawdown (%),Historical VaR 95% (%)
AAPL,0.106,28.675,-38.516,2.707
GOOGL,0.104,29.029,-44.320,2.693
MSFT,0.103,27.550,-37.148,2.657
AMZN,0.116,33.039,-56.145,3.061


## Set portfolio weights

In [18]:
portfolio_weights = pd.Series(
    1 / len(prices.columns),
    index=prices.columns,
    name="Weight"
)

display((portfolio_weights * 100).rename("Weight (%)").to_frame().round(2))

print("Total portfolio allocation:", f"{portfolio_weights.sum() * 100:.2f}%")
#For now, this simulator invests equally in all four stocks. You can change the weights

,Weight (%)
AAPL,25.0
GOOGL,25.0
MSFT,25.0
AMZN,25.0


Total portfolio allocation: 100.00%


## Calculate portfolio growth

In [19]:
portfolio_returns = (
    returns[portfolio_weights.index]
    .fillna(0)
    .mul(portfolio_weights, axis=1)
    .sum(axis=1)
)

portfolio_growth = INITIAL_INVESTMENT * (1 + portfolio_returns).cumprod()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=portfolio_growth.index,
    y=portfolio_growth,
    mode="lines",
    name="Portfolio Value"
))

fig.add_hline(
    y=INITIAL_INVESTMENT,
    line_dash="dash",
    annotation_text="Initial Investment"
)

fig.update_layout(
    title="Hypothetical Portfolio Growth",
    xaxis_title="Date",
    yaxis_title="Portfolio Value (USD)",
    template="plotly_dark",
    hovermode="x unified",
    height=600
)

show_chart(fig)

## Portfolio performance metrics
#### The Sharpe ratio compares returns with volatility, using your configured risk-free rate. These results are historical and hypothetical.

In [20]:
portfolio_daily_mean = portfolio_returns.mean()
portfolio_daily_std = portfolio_returns.std()

total_return = (
    portfolio_growth.iloc[-1] / INITIAL_INVESTMENT - 1
) * 100

elapsed_years = (
    (portfolio_growth.index[-1] - portfolio_growth.index[0]).days / 365.25
)

if elapsed_years > 0 and portfolio_growth.iloc[-1] > 0:
    annualized_return = (
        (portfolio_growth.iloc[-1] / INITIAL_INVESTMENT)
        ** (1 / elapsed_years) - 1
    ) * 100
else:
    annualized_return = np.nan

portfolio_volatility = portfolio_daily_std * np.sqrt(TRADING_DAYS) * 100

if portfolio_daily_std > 0:
    sharpe_ratio = (
        (portfolio_daily_mean * TRADING_DAYS - RISK_FREE_RATE)
        / (portfolio_daily_std * np.sqrt(TRADING_DAYS))
    )
else:
    sharpe_ratio = np.nan

portfolio_max_drawdown = (
    portfolio_growth / portfolio_growth.cummax() - 1
).min() * 100

portfolio_metrics = pd.DataFrame({
    "Metric": [
        "Initial Investment (USD)",
        "Final Portfolio Value (USD)",
        "Total Return (%)",
        "Annualized Return (%)",
        "Annualized Volatility (%)",
        "Sharpe Ratio",
        "Maximum Drawdown (%)"
    ],
    "Value": [
        INITIAL_INVESTMENT,
        portfolio_growth.iloc[-1],
        total_return,
        annualized_return,
        portfolio_volatility,
        sharpe_ratio,
        portfolio_max_drawdown
    ]
})

display(portfolio_metrics.round(3))

,Metric,Value
0,Initial Investment (USD),10000.000
1,Final Portfolio Value (USD),168305.221
2,Total Return (%),1583.052
3,Annualized Return (%),27.100
4,Annualized Volatility (%),24.724
5,Sharpe Ratio,1.095
6,Maximum Drawdown (%),-39.066


## Monte Carlo simulation
#### This creates 1,000 hypothetical one-year portfolio paths based on the portfolio's historical daily return distribution, using a simple normal-return assumption.

In [21]:
rng = np.random.default_rng(42)

daily_mean = portfolio_returns.mean()
daily_std = portfolio_returns.std()

simulated_daily_returns = rng.normal(
    loc=daily_mean,
    scale=daily_std,
    size=(TRADING_DAYS, NUM_SIMULATIONS)
)

simulated_paths = INITIAL_INVESTMENT * np.cumprod(
    1 + simulated_daily_returns,
    axis=0
)

simulation_dates = pd.bdate_range(
    start=prices.index[-1],
    periods=TRADING_DAYS + 1
)[1:]

simulation_df = pd.DataFrame(
    simulated_paths,
    index=simulation_dates,
    columns=[f"Simulation_{i + 1}" for i in range(NUM_SIMULATIONS)]
)

print("Simulations created:", NUM_SIMULATIONS)
print("Trading days simulated:", TRADING_DAYS)

Simulations created: 1000
Trading days simulated: 252


## Monte Carlo scenarios

In [22]:
fig = go.Figure()

number_to_plot = min(50, NUM_SIMULATIONS)

for i in range(number_to_plot):
    fig.add_trace(go.Scatter(
        x=simulation_df.index,
        y=simulation_df.iloc[:, i],
        mode="lines",
        line=dict(width=1),
        opacity=0.35,
        showlegend=False
    ))

fig.add_hline(
    y=INITIAL_INVESTMENT,
    line_dash="dash",
    annotation_text="Initial Investment"
)

fig.update_layout(
    title=f"Monte Carlo Portfolio Scenarios ({number_to_plot} Paths Shown)",
    xaxis_title="Future Date",
    yaxis_title="Simulated Portfolio Value (USD)",
    template="plotly_dark",
    height=600
)

show_chart(fig)

## Summarize Monte Carlo results


In [23]:
final_simulated_values = simulation_df.iloc[-1]

simulation_summary = pd.DataFrame({
    "Metric": [
        "Median Final Value",
        "Mean Final Value",
        "5th Percentile",
        "95th Percentile",
        "Probability of Ending Below Initial Investment"
    ],
    "Value (USD or Probability)": [
        final_simulated_values.median(),
        final_simulated_values.mean(),
        final_simulated_values.quantile(0.05),
        final_simulated_values.quantile(0.95),
        (final_simulated_values < INITIAL_INVESTMENT).mean()
    ]
})

display(simulation_summary.round(3))

,Metric,Value (USD or Probability)
0,Median Final Value,12638.834
1,Mean Final Value,13126.833
2,5th Percentile,8560.824
3,95th Percentile,18916.252
4,Probability of Ending Below Initial Investment,0.173


#### The last metric is a probability, so its value ranges from 0 to 1. Convert it to a percentage if wanted.

## Plot the distribution of simulated final values

In [24]:
fig = go.Figure(go.Histogram(
    x=final_simulated_values,
    nbinsx=40,
    name="Final Values"
))

fig.add_vline(
    x=INITIAL_INVESTMENT,
    line_dash="dash",
    annotation_text="Initial Investment"
)

fig.update_layout(
    title="Distribution of Simulated One-Year Portfolio Values",
    xaxis_title="Final Portfolio Value (USD)",
    yaxis_title="Number of Simulations",
    template="plotly_dark",
    height=500
)

show_chart(fig)

## Generate automatic insights

In [25]:
best_stock = cumulative_returns.iloc[-1].idxmax()
worst_stock = cumulative_returns.iloc[-1].idxmin()
most_volatile = annualized_volatility.idxmax()
least_volatile = annualized_volatility.idxmin()
largest_drawdown_stock = maximum_drawdown.idxmin()
highest_var_stock = var_losses.idxmax()

print("TECHSTOCK INSIGHTS — AUTOMATIC FINDINGS")
print("=" * 45)

print(
    f"Best cumulative performer: {best_stock} "
    f"({cumulative_returns.iloc[-1][best_stock] * 100:.2f}%)"
)

print(
    f"Worst cumulative performer: {worst_stock} "
    f"({cumulative_returns.iloc[-1][worst_stock] * 100:.2f}%)"
)

print(
    f"Most volatile stock: {most_volatile} "
    f"({annualized_volatility[most_volatile]:.2f}% annualized)"
)

print(
    f"Least volatile stock: {least_volatile} "
    f"({annualized_volatility[least_volatile]:.2f}% annualized)"
)

print(
    f"Largest historical drawdown: {largest_drawdown_stock} "
    f"({maximum_drawdown[largest_drawdown_stock]:.2f}%)"
)

print(
    f"Highest historical VaR: {highest_var_stock} "
    f"({var_losses[highest_var_stock]:.2f}%)"
)

print(
    f"Portfolio total return: {total_return:.2f}%"
)

print(
    f"Portfolio annualized volatility: {portfolio_volatility:.2f}%"
)

print(
    "Monte Carlo probability of ending below the initial investment: "
    f"{(final_simulated_values < INITIAL_INVESTMENT).mean() * 100:.2f}%"
)

TECHSTOCK INSIGHTS — AUTOMATIC FINDINGS
Best cumulative performer: AMZN (1555.26%)
Worst cumulative performer: GOOGL (1246.01%)
Most volatile stock: AMZN (33.04% annualized)
Least volatile stock: MSFT (27.55% annualized)
Largest historical drawdown: AMZN (-56.15%)
Highest historical VaR: AMZN (3.06%)
Portfolio total return: 1583.05%
Portfolio annualized volatility: 24.72%
Monte Carlo probability of ending below the initial investment: 17.30%


## Stock performance summary

In [26]:
performance_summary = pd.DataFrame({
    "First Price (USD)": prices.iloc[0],
    "Latest Price (USD)": prices.iloc[-1],
    "Price Change (%)": (prices.iloc[-1] / prices.iloc[0] - 1) * 100,
    "Cumulative Return (%)": cumulative_returns.iloc[-1] * 100,
    "Annualized Volatility (%)": returns.std() * np.sqrt(TRADING_DAYS) * 100,
    "Maximum Drawdown (%)": (prices / prices.cummax() - 1).min() * 100
})

display(performance_summary.round(3))

,First Price (USD),Latest Price (USD),Price Change (%),Cumulative Return (%),Annualized Volatility (%),Maximum Drawdown (%)
AAPL,24.172,344.825,1326.561,1326.561,28.675,-38.516
GOOGL,26.228,353.030,1246.008,1246.008,29.029,-44.320
MSFT,39.607,537.080,1256.017,1256.017,27.550,-37.148
AMZN,15.426,255.340,1555.257,1555.257,33.039,-56.145


## Export results to CSV

In [27]:
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

prices.to_csv(OUTPUT_FOLDER / "historical_prices.csv")
returns.to_csv(OUTPUT_FOLDER / "daily_returns.csv")
cumulative_returns.to_csv(OUTPUT_FOLDER / "cumulative_returns.csv")
correlation_matrix.to_csv(OUTPUT_FOLDER / "correlation_matrix.csv")
risk_summary.to_csv(OUTPUT_FOLDER / "risk_summary.csv")
performance_summary.to_csv(OUTPUT_FOLDER / "performance_summary.csv")
portfolio_growth.rename("Portfolio_Value").to_csv(
    OUTPUT_FOLDER / "portfolio_growth.csv"
)
portfolio_metrics.to_csv(
    OUTPUT_FOLDER / "portfolio_metrics.csv",
    index=False
)
simulation_df.to_csv(OUTPUT_FOLDER / "monte_carlo_simulations.csv")

print("CSV exports saved to:", OUTPUT_FOLDER.resolve())

CSV exports saved to: C:\Users\hp\techstock_outputs


## Export everything to one Excel workbook

In [28]:
excel_path = OUTPUT_FOLDER / "TechStock_Insights_Report.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    prices.to_excel(writer, sheet_name="Historical Prices")
    returns.to_excel(writer, sheet_name="Daily Returns")
    cumulative_returns.to_excel(writer, sheet_name="Cumulative Returns")
    correlation_matrix.to_excel(writer, sheet_name="Correlation")
    risk_summary.to_excel(writer, sheet_name="Risk Summary")
    performance_summary.to_excel(writer, sheet_name="Performance")
    portfolio_metrics.to_excel(
        writer, sheet_name="Portfolio Metrics", index=False
    )
    portfolio_weights.rename("Weight").to_frame().to_excel(
        writer, sheet_name="Portfolio Weights"
    )
    simulation_summary.to_excel(
        writer, sheet_name="Monte Carlo Summary", index=False
    )

print("Excel report saved to:", excel_path.resolve())

Excel report saved to: C:\Users\hp\techstock_outputs\TechStock_Insights_Report.xlsx


## Final verification

In [29]:
print("TECHSTOCK INSIGHTS — FINAL CHECK")
print("=" * 40)

print("Stocks analyzed:", ", ".join(prices.columns))
print("First date:", prices.index.min().date())
print("Last date:", prices.index.max().date())
print("Price rows:", len(prices))
print("Missing price values:", int(prices.isna().sum().sum()))
print("Daily return rows:", len(returns))
print("Portfolio final value:", f"${portfolio_growth.iloc[-1]:,.2f}")
print("Excel report exists:", excel_path.exists())

print("\nFiles created:")
for file_path in sorted(OUTPUT_FOLDER.iterdir()):
    if file_path.is_file():
        print(f"- {file_path.name} ({file_path.stat().st_size:,} bytes)")

TECHSTOCK INSIGHTS — FINAL CHECK
Stocks analyzed: AAPL, GOOGL, MSFT, AMZN
First date: 2015-01-02
Last date: 2026-10-14
Price rows: 2963
Missing price values: 0
Daily return rows: 2962
Portfolio final value: $168,305.22
Excel report exists: True

Files created:
- cleaned_stock_prices.csv (251,246 bytes)
- correlation_matrix.csv (290 bytes)
- cumulative_returns.csv (256,207 bytes)
- daily_returns.csv (289,302 bytes)
- historical_prices.csv (251,246 bytes)
- historical_stock_data.csv (1,006,889 bytes)
- monte_carlo_simulations.csv (4,686,527 bytes)
- performance_summary.csv (597 bytes)
- portfolio_growth.csv (90,117 bytes)
- portfolio_metrics.csv (292 bytes)
- risk_summary.csv (424 bytes)
- TechStock_Insights_Report.xlsx (526,184 bytes)
